# TrustOCT — Training Notebook 2: Attention Models
### Models: `msf_cbam_resnet50` + `msf_cbam_mixstyle_resnet50`
---
Run this notebook on **Account 2**. After training completes, results are synced to shared Google Drive.

In [ ]:
import os

if not os.path.exists('src'):
    !git clone https://github.com/Gnanapravallika/Trusthworthy_OCTAI.git
    %cd Trusthworthy_OCTAI
else:
    print('Repository already cloned.')

try:
    import albumentations
    import ptflops
except ImportError:
    !pip install -r requirements.txt

In [ ]:
import os
import sys
import time
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
from PIL import Image
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Running on device: {device}')

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Running locally. Skipping Google Drive mount.')

In [ ]:
if not os.path.exists('/content/Kermany') and not os.path.exists('/content/OCT2017'):
    try:
        from google.colab import files
        print("Please upload your kaggle.json API token file:")
        uploaded = files.upload()
        if 'kaggle.json' in uploaded:
            !mkdir -p ~/.kaggle
            !cp kaggle.json ~/.kaggle/
            !chmod 600 ~/.kaggle/kaggle.json
            print("Downloading Kermany OCT2017 dataset from Kaggle...")
            !kaggle datasets download -d paultimothymooney/kermany2018 --unzip -p /content/Kermany
            print("Dataset downloaded successfully.")
    except Exception as e:
        print(f"Download skipped: {e}")

In [ ]:
epochs = 30
lr = 1e-4
batch_size = 32

### 3. Train `msf_cbam_resnet50` (+ MultiScale + CBAM)

In [ ]:
from src.training.trainer import run_experiment
run_experiment('msf_cbam_resnet50', train_df, val_df, test_df, epochs=epochs, lr=lr, batch_size=batch_size, device_name=str(device))

### 4. Train `msf_cbam_mixstyle_resnet50` (+ MultiScale + CBAM + MixStyle)

In [ ]:
from src.training.trainer import run_experiment
run_experiment('msf_cbam_mixstyle_resnet50', train_df, val_df, test_df, epochs=epochs, lr=lr, batch_size=batch_size, device_name=str(device))

### Save Results to GitHub

In [ ]:
import subprocess

# --- PASTE YOUR GITHUB TOKEN BELOW ---
GITHUB_TOKEN = 'ghp_xxxxxxxxxxxxxxxxxxxx'  # Replace with your actual token
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/Gnanapravallika/Trusthworthy_OCTAI.git'

# Configure git identity (required for commits in Colab)
!git config user.email 'colab@trustoct.ai'
!git config user.name 'TrustOCT Colab'

# Stage output folders
!git add outputs/msf_cbam_resnet50/ outputs/msf_cbam_mixstyle_resnet50/ 2>/dev/null || true

# Check if there are staged changes
result = subprocess.run(['git', 'diff', '--cached', '--quiet'], capture_output=True)
if result.returncode != 0:
    !git commit -m 'NB2: Save msf_cbam_resnet50 + msf_cbam_mixstyle_resnet50 outputs'
    import shlex
    !git push $REPO_URL main
    print('Results successfully pushed to GitHub!')
else:
    print('No new changes to push.')